# Day 18: LangGraph Deep Dive — State Machines for Agent Orchestration

**30-Day AI Agent Challenge: OpenAI Agents SDK vs LangGraph vs CrewAI**

---

## Today's Concept

> Day 17's CrewAI crew was sequential: no loops, no branches. What happens when the Critic says "revise" and you actually want to act on it?

Today we rebuild Day 17's 5-role content crew (Ideator -> Hook -> Carousel -> Caption -> Critic) as a LangGraph state machine. The new capability: when the Critic returns REVISE, the graph routes the revision back to the **specific** weak node - the hooks, the carousel, or the caption. That targeted routing is the LangGraph lesson. CrewAI's sequential process cannot do it.

## What You'll Build
- The Day 17 content crew, rebuilt as a LangGraph graph
- A Critic that outputs a structured verdict (SHIP / REVISE + which element is weakest)
- A conditional edge that routes a REVISE to the specific weak node, not a dumb loop
- A circuit breaker so the graph always terminates


In [3]:
# ── Setup ───────────────────────────────────────────────
import sys
sys.path.insert(0, "..")
from shared import check_and_report, print_config, disable_openai_tracing, get_openai_agents_model, get_langgraph_model, get_crewai_llm
check_and_report()
disable_openai_tracing()
print("=" * 50)
print("DAY 18 SETUP COMPLETE")
print("=" * 50)
print_config()

Python: 3.12.13

All required packages installed.
Optional (missing): langchain-openai, langgraph-checkpoint-memory, ipywidgets

Ollama: connected (10 model(s) available)

DAY 18 SETUP COMPLETE
  Provider: OLLAMA
  Model:    qwen2.5:3b
  Ollama:   http://localhost:11434


## Today's Graph Structure

```
START -> ideator -> hook -> carousel -> caption -> critic --SHIP--> publish -> END
                ^          ^           ^                   |
                |          |           |              REVISE
                +----------+-----------+---- routes to the weakest element
```

Three producer nodes can all receive a revision. The Critic's verdict picks which one.
A revision counter caps the loop so the graph always terminates.


---
## LangGraph: Content Crew as a State Machine

Five nodes, one conditional edge, one circuit breaker. In CrewAI (Day 17) this crew
was strictly sequential - the Critic emitted a verdict and the crew ended. Here the
verdict drives the graph.

Note: in LangGraph every node sees the **full state**. There is no per-task
`context=[...]` like CrewAI. The tradeoff is simpler wiring (no context lists) and
more exposure (the Critic could snoop the raw angles if you let it - here it does not,
by prompt design).


In [5]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from typing_extensions import TypedDict

model = get_langgraph_model()


class ContentState(TypedDict):
    topic: str
    angles: str        # ideator output
    hooks: str         # hook writer output
    carousel: str      # carousel outliner output
    caption: str       # caption writer output
    critique: str      # critic full output
    verdict: str       # SHIP or REVISE
    target_node: str   # where to route on REVISE
    revision_count: int
    final_package: str


# ── Nodes (prompts mirror Day 17's roles so the comparison is honest) ──

def ideator(state: ContentState) -> dict:
    r = model.invoke([
        SystemMessage(content=(
            "You are a content strategist. You know the first idea is usually cliche, "
            "the second is safer, the third is where the insight lives. "
            "Output angles, not sentences."
        )),
        HumanMessage(content=(
            f"Generate 3 distinct angles for this topic. "
            f"Each angle = one sentence + one reason it lands.\n\n"
            f"Topic: {state['topic']}"
        )),
    ])
    return {"angles": r.content}


def hook_writer(state: ContentState) -> dict:
    r = model.invoke([
        SystemMessage(content=(
            "You write hooks like a copywriter, not a blogger. "
            "Specificity, contradiction, a number, a cost in the first 10 words. "
            "No setup sentence. No 'In today's world'."
        )),
        HumanMessage(content=(
            f"Pick the strongest angle. Write 5 hook variants for the first line. "
            f"Each hook 12 words or fewer.\n\n"
            f"Angles:\n{state['angles']}"
        )),
    ])
    return {"hooks": r.content}


def carousel_outliner(state: ContentState) -> dict:
    r = model.invoke([
        SystemMessage(content=(
            "You design swipeable carousels. Each slide title is one promise. "
            "Slide 1 is the hook restated visually, slides 2-6 are the payload, "
            "the last slide is the CTA."
        )),
        HumanMessage(content=(
            f"Design a 6-8 slide carousel outline. Output: slide number + title, one line each. "
            f"Slide 1 = hook, last = CTA.\n\n"
            f"Angles:\n{state['angles']}\n\nHooks:\n{state['hooks']}"
        )),
    ])
    return {"carousel": r.content}


def caption_writer(state: ContentState) -> dict:
    r = model.invoke([
        SystemMessage(content=(
            "You write captions that earn the save, not just the like. "
            "Short sentences. One idea per paragraph. "
            "A CTA that asks for a specific action, not 'thoughts?'."
        )),
        HumanMessage(content=(
            f"Write the caption (120 words or fewer). Body + one specific CTA + 3-5 hashtags. "
            f"Match the hook's tone.\n\n"
            f"Hooks:\n{state['hooks']}\n\nCarousel:\n{state['carousel']}"
        )),
    ])
    return {"caption": r.content}


def critic(state: ContentState) -> dict:
    """Review the package. Output SHIP/REVISE and name the weakest element.
    Parsed as strict text (not structured output) because a 3B local model is more
    reliable with a fixed text format than with function-calling."""
    r = model.invoke([
        SystemMessage(content=(
            "You are a senior content critic. You never say 'looks good'. "
            "Review the package and output EXACTLY three lines, no preamble:\n"
            "VERDICT: SHIP or REVISE\n"
            "WEAKEST: hooks or carousel or caption\n"
            "FIX: the one change that would move this from scannable to save-worthy"
        )),
        HumanMessage(content=(
            f"Hooks:\n{state['hooks']}\n\n"
            f"Carousel:\n{state['carousel']}\n\n"
            f"Caption:\n{state['caption']}"
        )),
    ])
    text = r.content
    verdict = "SHIP"
    target = "caption"
    for line in text.splitlines():
        low = line.lower()
        if low.startswith("verdict:"):
            verdict = "SHIP" if "ship" in low else "REVISE"
        elif low.startswith("weakest:"):
            if "hook" in low:
                target = "hook"
            elif "carousel" in low:
                target = "carousel"
            elif "caption" in low:
                target = "caption"
    return {
        "critique": text,
        "verdict": verdict,
        "target_node": target,
        "revision_count": state["revision_count"] + 1,
    }


def route_after_critic(state: ContentState) -> str:
    # Circuit breaker: force publish after 2 critic passes so the graph always terminates.
    if state["revision_count"] >= 2:
        return "publish"
    if state["verdict"] == "SHIP":
        return "publish"
    # REVISE: route to the specific weak node, not a dumb loop.
    return state["target_node"]


def publish(state: ContentState) -> dict:
    package = (
        f"=== ANGLES ===\n{state['angles']}\n\n"
        f"=== HOOKS ===\n{state['hooks']}\n\n"
        f"=== CAROUSEL ===\n{state['carousel']}\n\n"
        f"=== CAPTION ===\n{state['caption']}\n\n"
        f"=== FINAL CRITIQUE ===\n{state['critique']}"
    )
    return {"final_package": package}


# ── Build the graph ────────────────────────────────────
builder = StateGraph(ContentState)
builder.add_node("ideator", ideator)
builder.add_node("hook", hook_writer)
builder.add_node("carousel", carousel_outliner)
builder.add_node("caption", caption_writer)
builder.add_node("critic", critic)
builder.add_node("publish", publish)

builder.add_edge(START, "ideator")
builder.add_edge("ideator", "hook")
builder.add_edge("hook", "carousel")
builder.add_edge("carousel", "caption")
builder.add_edge("caption", "critic")
builder.add_conditional_edges(
    "critic", route_after_critic,
    {"hook": "hook", "carousel": "carousel", "caption": "caption", "publish": "publish"},
)
builder.add_edge("publish", END)

graph = builder.compile()

print("=" * 60)
print("LANGGRAPH - CONTENT CREW WITH REVISION LOOP")
print("=" * 60)
print("Nodes:", list(graph.nodes.keys()))

# Run
result = graph.invoke({
    "topic": "Why every developer should learn AI agents in 2026",
    "angles": "", "hooks": "", "carousel": "", "caption": "",
    "critique": "", "verdict": "", "target_node": "",
    "revision_count": 0, "final_package": "",
})

print(f"\nCritic passes: {result['revision_count']}")
print(f"Final verdict: {result['verdict']}")
print(f"Last revision target: {result['target_node']}")
print(f"\nFINAL PACKAGE:")
print(result["final_package"])


LANGGRAPH - CONTENT CREW WITH REVISION LOOP
Nodes: ['__start__', 'ideator', 'hook', 'carousel', 'caption', 'critic', 'publish']

Critic passes: 2
Final verdict: REVISE
Last revision target: hook

FINAL PACKAGE:
=== ANGLES ===
Angle 1:
"One reason to lean AI agents is their potential to revolutionize backend development, making server-side coding more efficient and less error-prone."
Reason: Rapid advancements in machine learning are transforming the backend landscape. Understanding AI can equip developers with new tools for optimization and innovation.

Angle 2:
"Another angle is that integrating AI into applications ensures better user experiences through personalized features without compromising security protocols."
Reason: As AI becomes indispensable, users demand smarter interfaces. By mastering AI agents, developers can create more secure and intuitive solutions to meet these expectations.

Angle 3:
"A third perspective is on the future market landscape where AI skills will be a 

---
## Day 17 vs Day 18: Same Crew, Different Power

| Aspect | Day 17 (CrewAI sequential) | Day 18 (LangGraph graph) |
|---|---|---|
| Loop on REVISE | Not possible without Flows | One conditional edge |
| Route revision to specific node | Not possible | Critic picks the target |
| Circuit breaker | Not needed (no loop) | revision_count cap |
| Per-task context | `context=[task]` | Full state visible to every node |
| State inspection | CrewAI internals | TypedDict, fully visible |
| Ceremony cost | Low | Higher - but it buys the loop |

The tradeoff is honest. LangGraph costs more to write, and for a linear crew you would
not pay it (CrewAI was the right call on Day 17). You reach for LangGraph when the flow
has a loop, a branch, or a checkpoint. This crew has all three once the Critic can send
you back to any of three nodes.


## Key Takeaway

LangGraph earns its ceremony the moment your flow has a loop or a branch. Day 17's
content crew was sequential - CrewAI was the right tool. The same crew with a Critic
that can route a revision to any of three nodes is a graph - LangGraph is the right tool.

The targeted revision is the key idea. A dumb "try again" loop is cheap anywhere.
A loop that routes to the specific weak element, with a circuit breaker so it
terminates, is where `add_conditional_edges` pays off. In CrewAI you would need Flows
to get this.

One production note: the Critic here outputs strict text (VERDICT / WEAKEST / FIX)
parsed with string ops, not structured output via function-calling. On a 3B local
model, strict text is more reliable than function-calling. On a larger hosted model,
`with_structured_output` is cleaner. Pick the parsing strategy to match the model.

---
